# 3. Príznaky klipov pomocou video siete R(2+1)D

**R(2+1)D-18** je konvolučná sieť pre video, predtrénovaná na datasete **Kinetics-400**
(400 ľudských činností). Každú 3D konvolúciu rozkladá na 2D konvolúciu v priestore
a 1D konvolúciu v čase, takže spracúva celý klip naraz – vrátane pohybu medzi snímkami.

Vstup: 16 snímok, veľkosť 112 × 112, normalizácia Kinetics. Všetko robí `weights.transforms()`.

V tomto notebooku:
1. predpoveď siete bez trénovania,
2. príznaky klipov `features_r21d.npy` – tvar `(počet klipov, 512)`, pre notebook 6,
3. postupnosť príznakov `features_r21d_seq.npy` – tvar `(počet klipov, 4, 512)`, pre notebook 6c.

In [1]:
import numpy as np
import pandas as pd
import torch
from torch import nn
from torchvision.models.video import r2plus1d_18, R2Plus1D_18_Weights

from har.config import CLASSES, INDEX_FILE, KINETICS_CLASSES
from har.data import read_frames, to_video_tensor

df = pd.read_csv(INDEX_FILE)
device = "cuda" if torch.cuda.is_available() else "cpu"

weights = R2Plus1D_18_Weights.KINETICS400_V1
video_transform = weights.transforms()
model = r2plus1d_18(weights=weights).eval().to(device)

## Predpoveď bez trénovania

Sieť s pôvodnou poslednou vrstvou zaraďuje klip do 400 tried Kinetics.
Pre každú našu triedu sčítame pravdepodobnosti zodpovedajúcich tried Kinetics
(`KINETICS_CLASSES` v `har/config.py`) a vyberieme triedu s najväčším súčtom.

In [2]:
categories = weights.meta["categories"]
kinetics_ids = {cls: [categories.index(c) for c in KINETICS_CLASSES[cls]] for cls in CLASSES}

rows = []
with torch.no_grad():
    for path, cls in zip(df.path, df.cls):
        x = to_video_tensor(read_frames(path), video_transform).to(device)
        prob = model(x).softmax(1)[0].cpu()
        row = {"cls": cls, "top1": categories[prob.argmax()]}
        row |= {c: prob[ids].sum().item() for c, ids in kinetics_ids.items()}
        rows.append(row)
zero_shot = pd.DataFrame(rows)

for cls, g in zero_shot.groupby("cls"):
    print(f"{cls}: najčastejšie triedy Kinetics", g.top1.value_counts().head(5).to_dict())

pred = zero_shot[CLASSES].idxmax(axis=1)
print(f"presnosť bez trénovania: {(pred == zero_shot.cls).mean():.0%}")

clap: najčastejšie triedy Kinetics {'clapping': 15, 'sign language interpreting': 15, 'applauding': 12, 'celebrating': 8, 'air drumming': 6}
eat: najčastejšie triedy Kinetics {'eating spaghetti': 10, 'dining': 9, 'smoking': 8, 'eating burger': 7, 'eating chips': 7}
presnosť bez trénovania: 90%


## Príznaky klipov

Poslednú vrstvu `fc` (400 tried Kinetics) nahradíme `nn.Identity()` – výstupom je
**jeden vektor 512 príznakov na celý klip**. Sieť sa netrénuje.

In [3]:
model.fc = nn.Identity()

feats = []
with torch.no_grad():
    for path in df.path:
        x = to_video_tensor(read_frames(path), video_transform).to(device)   # (1, 3, 16, 112, 112)
        feats.append(model(x)[0].cpu().numpy())                              # (512,)

feats = np.stack(feats)
np.save("features_r21d.npy", feats)
print(feats.shape)

(238, 512)


## Postupnosť príznakov

Pre model R(2+1)D + LSTM potrebujeme namiesto jedného vektora na klip **postupnosť** vektorov.
Z klipu vyberieme 32 rovnomerne rozložených snímok, rozdelíme ich na 4 úseky po 8 snímok
a každý úsek prevedieme sieťou na 512 príznakov → `(4, 512)` na klip.

In [4]:
N_CHUNKS, CHUNK = 4, 8

seq = []
with torch.no_grad():
    for path in df.path:
        frames = read_frames(path, n=N_CHUNKS * CHUNK)                       # (32, 224, 224, 3)
        x = torch.cat([to_video_tensor(frames[i * CHUNK:(i + 1) * CHUNK], video_transform)
                       for i in range(N_CHUNKS)]).to(device)                 # (4, 3, 8, 112, 112)
        seq.append(model(x).cpu().numpy())                                   # (4, 512)

seq = np.stack(seq)
np.save("features_r21d_seq.npy", seq)
print(seq.shape)

(238, 4, 512)
